# Item Similarity Module



### Goal of this notebook
1. Build a **content-based** representation of every movie (genres + tags)
2. Calculate an **item-item similarity matrix** using Cosine Similarity
3. Build a `get_similar_movies()` function 
4. Save the similarity


##  Imports

In [1]:
import pandas as pd
import numpy as np
import pickle
import os

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

PROCESSED_DIR = 'processed'
MODEL_DIR = 'models'
os.makedirs(MODEL_DIR, exist_ok=True)


## Load Movies Metadata

In [ ]:
movies_full = pd.read_csv(os.path.join(PROCESSED_DIR, 'movies_full.csv')) 
print("Movies shape:", movies_full.shape)
movies_full.head()


Movies shape: (9742, 9)


,movieId,title,genres,year,clean_title,genres_list,all_tags,imdbId,tmdbId
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1995.0,Toy Story,"['Adventure', 'Animation', 'Children', 'Comedy...",pixar pixar fun,114709.0,862.0
1,2,Jumanji (1995),Adventure|Children|Fantasy,1995.0,Jumanji,"['Adventure', 'Children', 'Fantasy']",fantasy magic board game Robin Williams game,113497.0,8844.0
2,3,Grumpier Old Men (1995),Comedy|Romance,1995.0,Grumpier Old Men,"['Comedy', 'Romance']",moldy old,113228.0,15602.0
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,1995.0,Waiting to Exhale,"['Comedy', 'Drama', 'Romance']",NaN,114885.0,31357.0
4,5,Father of the Bride Part II (1995),Comedy,1995.0,Father of the Bride Part II,['Comedy'],pregnancy remake,113041.0,11862.0


In [ ]:
movies_full[['movieId', 'clean_title', 'genres', 'all_tags']].head(10)
# We will see that genres in movies_full is pipe-separated and all_tags is free text.

,movieId,clean_title,genres,all_tags
0,1,Toy Story,Adventure|Animation|Children|Comedy|Fantasy,pixar pixar fun
1,2,Jumanji,Adventure|Children|Fantasy,fantasy magic board game Robin Williams game
2,3,Grumpier Old Men,Comedy|Romance,moldy old
3,4,Waiting to Exhale,Comedy|Drama|Romance,NaN
4,5,Father of the Bride Part II,Comedy,pregnancy remake
5,6,Heat,Action|Crime|Thriller,NaN
6,7,Sabrina,Comedy|Romance,remake
7,8,Tom and Huck,Adventure|Children,NaN
8,9,Sudden Death,Action,NaN
9,10,GoldenEye,Action|Adventure|Thriller,NaN


##  Build the trick for Each Movie

We combine **genres** + **tags** into one text string per movie. This is trick of content-based recommender :

So we will merge all descriptive text into a single "document" per item, then vectorize it.



In [ ]:
def build_trick(row):
    genres_text = str(row['genres']).replace('|', ' ').replace('(no genres listed)', '')
    tags_text = str(row['all_tags']) if pd.notna(row['all_tags']) else ''

    # Repeat genres twice to give them more weight than tags
    soup = (genres_text + ' ') * 2 + tags_text
    return soup.strip().lower()


movies_full['full_content'] = movies_full.apply(build_trick, axis=1)
movies_full[['clean_title', 'full_content']].head(10)


,clean_title,content_soup
0,Toy Story,adventure animation children comedy fantasy ad...
1,Jumanji,adventure children fantasy adventure children ...
2,Grumpier Old Men,comedy romance comedy romance moldy old
3,Waiting to Exhale,comedy drama romance comedy drama romance
4,Father of the Bride Part II,comedy comedy pregnancy remake
5,Heat,action crime thriller action crime thriller
6,Sabrina,comedy romance comedy romance remake
7,Tom and Huck,adventure children adventure children
8,Sudden Death,action action
9,GoldenEye,action adventure thriller action adventure thr...


In [ ]:
# Sanity check: how many movies have an empty content soup (no genres AND no tags)?
empty_soup_count = (movies_full['full_content'].str.strip() == '').sum()
print(f"Movies with empty content soup: {empty_soup_count} out of {len(movies_full)}")


Movies with empty content soup: 33 out of 9742


## 4. TF-IDF Vectorization

Converts each movie's content soup into a numeric vector, where each dimension
represents a genre/tag term, weighted by how informative that term is across
the whole movie catalog.


In [ ]:
tfidf = TfidfVectorizer(stop_words='english', min_df=1)
tfidf_matrix = tfidf.fit_transform(movies_full['full_content'])

print("TF-IDF matrix shape:", tfidf_matrix.shape)
print("Vocabulary size:", len(tfidf.vocabulary_))


TF-IDF matrix shape: (9742, 1675)
Vocabulary size: 1675


## 5. Cosine Similarity Matrix

From the lecture: **Cosine Similarity** measures the angle between two vectors —
the smaller the angle, the more similar the items. It's ideal here because it's
not affected by vector length (i.e., movies with more tags aren't unfairly favored).

⚠️ **Note on scalability:** Computing the full N×N similarity matrix works fine
for MovieLens-small (~9,700 movies → ~94M float entries, manageable in memory).
For a much larger catalog, you'd switch to on-demand similarity (compute row-by-row)
or approximate nearest-neighbor search (e.g., FAISS).


In [7]:
# Compute full cosine similarity matrix (movies x movies)
cosine_sim_matrix = cosine_similarity(tfidf_matrix, tfidf_matrix)

print("Cosine similarity matrix shape:", cosine_sim_matrix.shape)
print("Memory size (approx):", cosine_sim_matrix.nbytes / (1024**2), "MB")


Cosine similarity matrix shape: (9742, 9742)
Memory size (approx): 724.0796203613281 MB


In [8]:
# Build a mapping from movieId -> row index in the similarity matrix
movieId_to_row = {mid: idx for idx, mid in enumerate(movies_full['movieId'])}
row_to_movieId = {idx: mid for mid, idx in movieId_to_row.items()}


## 6. `get_similar_movies()` — The Core Function for the Item Page

Given a `movieId`, returns the **Top-N most similar movies** (excluding itself).
This is exactly what the dashboard's Item Page will call when a user selects a movie.


In [9]:
def get_similar_movies(movie_id, n=10, sim_matrix=cosine_sim_matrix,
                        movieId_to_row=movieId_to_row, row_to_movieId=row_to_movieId,
                        movies_df=movies_full):
    """
    Returns a DataFrame of the Top-N movies most similar to the given movieId,
    sorted by similarity score (descending), excluding the movie itself.
    """
    if movie_id not in movieId_to_row:
        raise ValueError(f"movieId {movie_id} not found in the catalog.")

    row_idx = movieId_to_row[movie_id]
    sim_scores = sim_matrix[row_idx]

    # Get indices sorted by similarity, descending, skip the first one (itself)
    similar_indices = np.argsort(sim_scores)[::-1]
    similar_indices = similar_indices[similar_indices != row_idx][:n]

    result_movie_ids = [row_to_movieId[idx] for idx in similar_indices]
    result_scores = [sim_scores[idx] for idx in similar_indices]

    result_df = movies_df[movies_df['movieId'].isin(result_movie_ids)].copy()
    # Preserve similarity order
    result_df['similarity_score'] = result_df['movieId'].map(dict(zip(result_movie_ids, result_scores)))
    result_df = result_df.sort_values('similarity_score', ascending=False).reset_index(drop=True)

    return result_df[['movieId', 'clean_title', 'genres', 'year', 'similarity_score']]


In [10]:
# Test it on a well-known movie (Toy Story, movieId=1, if present in your dataset)
sample_movie_id = movies_full['movieId'].iloc[0]
sample_title = movies_full[movies_full['movieId'] == sample_movie_id]['clean_title'].values[0]

print(f"Movies similar to: '{sample_title}' (movieId={sample_movie_id})\n")
similar = get_similar_movies(sample_movie_id, n=10)
similar


Movies similar to: 'Toy Story' (movieId=1)



,movieId,clean_title,genres,year,similarity_score
0,2355,"Bug's Life, A",Adventure|Animation|Children|Comedy,1998.0,0.840982
1,3114,Toy Story 2,Adventure|Animation|Children|Comedy|Fantasy,1999.0,0.725594
2,2294,Antz,Adventure|Animation|Children|Comedy|Fantasy,1998.0,0.608402
3,3754,"Adventures of Rocky and Bullwinkle, The",Adventure|Animation|Children|Comedy|Fantasy,2000.0,0.608402
4,45074,"Wild, The",Adventure|Animation|Children|Comedy|Fantasy,2006.0,0.608402
5,53121,Shrek the Third,Adventure|Animation|Children|Comedy|Fantasy,2007.0,0.608402
6,91355,Asterix and the Vikings (Astérix et les Vikings),Adventure|Animation|Children|Comedy|Fantasy,2006.0,0.608402
7,103755,Turbo,Adventure|Animation|Children|Comedy|Fantasy,2013.0,0.608402
8,136016,The Good Dinosaur,Adventure|Animation|Children|Comedy|Fantasy,2015.0,0.608402
9,166461,Moana,Adventure|Animation|Children|Comedy|Fantasy,2016.0,0.608402


In [11]:
# Try another random sample to spot-check quality
sample_movie_id_2 = movies_full['movieId'].sample(1, random_state=7).values[0]
sample_title_2 = movies_full[movies_full['movieId'] == sample_movie_id_2]['clean_title'].values[0]

print(f"Movies similar to: '{sample_title_2}' (movieId={sample_movie_id_2})\n")
get_similar_movies(sample_movie_id_2, n=10)


Movies similar to: 'Hit the Bank (Vabank)' (movieId=5912)



,movieId,clean_title,genres,year,similarity_score
0,63,Don't Be a Menace to South Central While Drink...,Comedy|Crime,1996.0,1.0
1,4395,Big Deal on Madonna Street (I Soliti Ignoti),Comedy|Crime,1958.0,1.0
2,4662,"See No Evil, Hear No Evil",Comedy|Crime,1989.0,1.0
3,5900,Analyze That,Comedy|Crime,2002.0,1.0
4,6572,Scorched,Comedy|Crime,2003.0,1.0
5,6663,"Pink Panther Strikes Again, The",Comedy|Crime,1976.0,1.0
6,6763,Duplex,Comedy|Crime,2003.0,1.0
7,31123,Ruby & Quentin (Tais-toi!),Comedy|Crime,2003.0,1.0
8,93022,Miss Nobody,Comedy|Crime,2010.0,1.0
9,97306,Seven Psychopaths,Comedy|Crime,2012.0,1.0


## 7. Pagination Helper (for the Item Page Navigation)

The Item Page needs "next / previous / specific page" navigation through the
similar-items list. We compute a larger pool (e.g. top 50) once, then paginate
through it in memory — fast, and no recomputation needed when the user clicks "next".


In [12]:
def get_similar_movies_page(movie_id, page=1, page_size=10, pool_size=50):
    """
    Returns one 'page' of the similar-movies list, plus pagination metadata.
    """
    full_pool = get_similar_movies(movie_id, n=pool_size)
    total_items = len(full_pool)
    total_pages = max(1, (total_items + page_size - 1) // page_size)

    page = max(1, min(page, total_pages))  # clamp to valid range
    start = (page - 1) * page_size
    end = start + page_size

    return {
        'data': full_pool.iloc[start:end].reset_index(drop=True),
        'current_page': page,
        'total_pages': total_pages,
        'page_size': page_size,
        'total_items': total_items
    }


# Test pagination
result = get_similar_movies_page(sample_movie_id, page=1, page_size=5)
print(f"Page {result['current_page']} of {result['total_pages']} (showing {result['page_size']} of {result['total_items']})")
result['data']


Page 1 of 10 (showing 5 of 50)


,movieId,clean_title,genres,year,similarity_score
0,2355,"Bug's Life, A",Adventure|Animation|Children|Comedy,1998.0,0.840982
1,3114,Toy Story 2,Adventure|Animation|Children|Comedy|Fantasy,1999.0,0.725594
2,4016,"Emperor's New Groove, The",Adventure|Animation|Children|Comedy|Fantasy,2000.0,0.608402
3,4886,"Monsters, Inc.",Adventure|Animation|Children|Comedy|Fantasy,2001.0,0.608402
4,3754,"Adventures of Rocky and Bullwinkle, The",Adventure|Animation|Children|Comedy|Fantasy,2000.0,0.608402


## 8. Save Artifacts for the Dashboard

We save:
1. The **TF-IDF vectorizer** (in case we want to vectorize new movies later)
2. The **cosine similarity matrix** (the heavy computation — precomputed once!)
3. The **movieId ↔ row index mappings**

This means the dashboard **never recomputes similarity** — it just loads these
files and looks up results instantly when a user selects a movie.


In [13]:
import scipy.sparse as sp

# Save the similarity matrix (as a compressed numpy file - it's a dense matrix)
np.savez_compressed(os.path.join(MODEL_DIR, 'cosine_sim_matrix.npz'), sim_matrix=cosine_sim_matrix)

# Save the TF-IDF vectorizer
with open(os.path.join(MODEL_DIR, 'tfidf_vectorizer.pkl'), 'wb') as f:
    pickle.dump(tfidf, f)

# Save the row <-> movieId mappings
with open(os.path.join(MODEL_DIR, 'item_similarity_mappings.pkl'), 'wb') as f:
    pickle.dump({
        'movieId_to_row': movieId_to_row,
        'row_to_movieId': row_to_movieId
    }, f)

print("✅ Saved cosine_sim_matrix.npz")
print("✅ Saved tfidf_vectorizer.pkl")
print("✅ Saved item_similarity_mappings.pkl")


✅ Saved cosine_sim_matrix.npz
✅ Saved tfidf_vectorizer.pkl
✅ Saved item_similarity_mappings.pkl


In [14]:
# Quick reload test - simulate what the dashboard will do
loaded = np.load(os.path.join(MODEL_DIR, 'cosine_sim_matrix.npz'))
reloaded_sim_matrix = loaded['sim_matrix']

with open(os.path.join(MODEL_DIR, 'item_similarity_mappings.pkl'), 'rb') as f:
    reloaded_mappings = pickle.load(f)

# Re-run get_similar_movies using the reloaded artifacts only
test_result = get_similar_movies(
    sample_movie_id, n=5,
    sim_matrix=reloaded_sim_matrix,
    movieId_to_row=reloaded_mappings['movieId_to_row'],
    row_to_movieId=reloaded_mappings['row_to_movieId']
)
print("Reload test successful! Results match expected format:")
test_result


Reload test successful! Results match expected format:


,movieId,clean_title,genres,year,similarity_score
0,2355,"Bug's Life, A",Adventure|Animation|Children|Comedy,1998.0,0.840982
1,3114,Toy Story 2,Adventure|Animation|Children|Comedy|Fantasy,1999.0,0.725594
2,91355,Asterix and the Vikings (Astérix et les Vikings),Adventure|Animation|Children|Comedy|Fantasy,2006.0,0.608402
3,103755,Turbo,Adventure|Animation|Children|Comedy|Fantasy,2013.0,0.608402
4,166461,Moana,Adventure|Animation|Children|Comedy|Fantasy,2016.0,0.608402


## 9. Phase 3 Summary

✅ Built a **content soup** per movie (genres weighted x2 + tags)
✅ Vectorized with **TF-IDF**
✅ Computed the full **Cosine Similarity matrix** between all movies
✅ Built `get_similar_movies(movie_id, n)` → core function for the Item Page
✅ Built `get_similar_movies_page(movie_id, page, page_size)` → handles pagination/navigation
✅ Saved precomputed artifacts so the dashboard loads instantly (no recomputation per click)
✅ Verified everything reloads and works correctly from saved files

### 📦 Output Files (in `models/` folder)
| File | Description |
|------|-------------|
| `cosine_sim_matrix.npz` | Precomputed movie-movie similarity matrix |
| `tfidf_vectorizer.pkl` | Fitted TF-IDF vectorizer (genres + tags) |
| `item_similarity_mappings.pkl` | movieId ↔ matrix row index mappings |

---
**Next Phase →** Phase 4: Dashboard - User Page (Streamlit).
